# DENTRACE — Level 3: DENTEX 4-Class Abnormal-Tooth Instance Segmentation
**Project:** DENTRACE — Local-First Dental OPG Anomaly Detection & Claim Engine  
**Team:** Team XYNPASE (Team ID 81), GLA University, Mathura  
**Regulatory Notice:** *Draft for dentist review. Not a diagnosis. Academic decision-support prototype.*

### Compulsory Target Classes:
1. `impacted_tooth` (Class 0)
2. `caries` (Class 1)
3. `periapical_lesion` (Class 2)
4. `deep_caries` (Class 3)

**Task Definition:** Abnormal-Tooth Instance Segmentation (segmenting full abnormal tooth contours harboring specific dental pathologies).

In [ ]:
# 1. Verify GPU Hardware
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics onnx onnxruntime albumentations

In [ ]:
# 3. Dataset Setup
# Upload or mount your verified yolo_seg.zip dataset
# Authorization Notice: Ensure dataset license compliance before uploading.
!mkdir -p dentex_yolo_seg
# !unzip -q /content/yolo_seg.zip -d dentex_yolo_seg/

# Verify data.yaml configuration
yaml_text = """
path: /content/dentex_yolo_seg
train: images/train
val: images/val

names:
  0: impacted_tooth
  1: caries
  2: periapical_lesion
  3: deep_caries

nc: 4
"""
with open('/content/dentex_yolo_seg/data.yaml', 'w') as f:
    f.write(yaml_text)
print("data.yaml configured successfully.")

In [ ]:
# 4. Train YOLOv8n-Seg Baseline
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='/content/dentex_yolo_seg/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # T4 GPU supports batch 16
    patience=10,       # Early stopping on validation metrics
    save=True,
    device=0,
    workers=4,
    seed=42,
    deterministic=True,
    half=True,         # FP16 mixed precision
    project='dentex_experiments',
    name='yolov8n_seg_4class',
    exist_ok=True
)

In [ ]:
# 5. Evaluate Validation Set Metrics
metrics = model.val()
print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")

for cls_idx, cls_name in enumerate(['impacted_tooth', 'caries', 'periapical_lesion', 'deep_caries']):
    if cls_idx < len(metrics.seg.maps):
        print(f"Class {cls_name} Mask AP50-95: {metrics.seg.maps[cls_idx]:.4f}")

In [ ]:
# 6. Export to ONNX for Parity Validation
best_weights = 'dentex_experiments/yolov8n_seg_4class/weights/best.pt'
export_model = YOLO(best_weights)
exported_path = export_model.export(format='onnx', imgsz=640, opset=17)
print(f"Model exported to ONNX: {exported_path}")

In [ ]:
# 7. Package Artifacts for Download to Local Environment
!zip -r /content/level3_yolov8n_seg_artifacts.zip dentex_experiments/yolov8n_seg_4class/weights/
from google.colab import files
files.download('/content/level3_yolov8n_seg_artifacts.zip')